## Training notebook

In [1]:
import gym
import highway_env

from stable_baselines3 import PPO, TD3
from sb3_contrib import RecurrentPPO

%load_ext tensorboard
from datetime import datetime
import os
from ipywidgets import interact, widgets
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback, BaseCallback

from typing import Callable

/home/utente1/miniconda3/envs/highwayvenv/lib/python3.8/site-packages/gym/envs/registration.py:216: UserWarning: WARN: Overriding environment __root__/HandManipulateBlockRotateZ-v1
  logger.warn("Overriding environment {}".format(id))
/home/utente1/miniconda3/envs/highwayvenv/lib/python3.8/site-packages/gym/envs/registration.py:216: UserWarning: WARN: Overriding environment __root__/HandManipulateBlockRotateZDense-v1
  logger.warn("Overriding environment {}".format(id))


In [2]:
%tensorboard --logdir /home/utente1/pighetti/HighwayDRL/training_output/logdir/

Reusing TensorBoard on port 6008 (pid 52660), started 0:04:26 ago. (Use '!kill 52660' to kill it.)

### Select operating system

In [3]:
os_id = widgets.Text()
def f(desired_os):
    os_id.value = str(desired_os)
interact(f, desired_os=['MACOS', 'WINDOWS', 'UBUNTU_MICRO', 'UBUNTU_DELL'])

interactive(children=(Dropdown(description='desired_os', options=('MACOS', 'WINDOWS', 'UBUNTU_MICRO', 'UBUNTU_…

<function __main__.f(desired_os)>

In [4]:
OUTPUT_PATH = '/Users/fornerispighetti/HighwayDRL/training_output/' if os_id.value == "MACOS" else 'C:/Users/pigo/Desktop/HighwayDRL/training_output/'
if os_id.value == 'UBUNTU_MICRO':
    OUTPUT_PATH = '/home/utente1/Desktop/PighettiForneris/HighwayDRL/training_output/'
elif os_id.value == 'UBUNTU_DELL':
    OUTPUT_PATH = '/home/utente1/pighetti/HighwayDRL/training_output'

In [5]:
# OUTPUT_PATH = '/home/elios/Tesi_FP/HighwayDRL/training_output/'

### Select environment

In [6]:
env_id = widgets.Text()
def f(input_env):
    env_id.value = str(input_env)
interact(f, input_env=['highway-v0','dm-env-v0','acc-dm-env-v0','jam-dm-env-v0','overtaken-dm-env-v0','singleO-dm-env-v0','multipleO-dm-env-v0'])

interactive(children=(Dropdown(description='input_env', options=('highway-v0', 'dm-env-v0', 'acc-dm-env-v0', '…

<function __main__.f(input_env)>

In [9]:
total_timesteps = 6e5
# Create and wrap the environment
env = gym.make(env_id.value)
env.configure({
    "training_total_timesteps": total_timesteps
})
# Create a TensorboardCallback to plot sub-rewards

class TensorboardCallback(BaseCallback):
    def __init__(self, verbose=1):
        super(TensorboardCallback, self).__init__(verbose)
        self.coll_rew = 0
        self.rml_rew = 0
        self.high_speed_rew = 0
        self.current_steps = 0

    def _on_step(self) -> bool:        
        self.rml_rew += self.training_env.get_attr('rml_reward')[0]
        self.high_speed_rew += self.training_env.get_attr('high_speed_reward')[0]
            
        if self.locals["dones"][0]:
            self.coll_rew += -self.training_env.get_attr('collision_reward')[0] * self.training_env.get_attr('vehicle')[0].crashed
            self.logger.record("eval/coll_rew", self.coll_rew)
            self.logger.record("episode/RML_rew", self.rml_rew / self.current_steps)
            self.logger.record("episode/high_speed_rew", self.high_speed_rew / self.current_steps)
            self.rml_rew = self.high_speed_rew = self.current_steps = 0
        else:
            self.current_steps = self.training_env.get_attr('steps')[0]
        return True

# Save a checkpoint every 5000 steps
checkpoint_callback = CheckpointCallback(save_freq=5000, save_path=f'{OUTPUT_PATH}/checkpoints',
                                         name_prefix='dm_rl_callback')

eval_callback = EvalCallback(env, best_model_save_path=f'{OUTPUT_PATH}/eval_logs',
                             log_path=f'{OUTPUT_PATH}/eval_logs', eval_freq=1000,
                             deterministic=True, render=False)

In [10]:
def linear_schedule(initial_value: float) -> Callable[[float], float]:
    """
    Linear learning rate schedule.

    :param initial_value: Initial learning rate.
    :return: schedule that computes
      current learning rate depending on remaining progress
    """
    def func(progress_remaining: float) -> float:
        """
        Progress will decrease from 1 (beginning) to 0.

        :param progress_remaining:
        :return: current learning rate
        """
        return progress_remaining * initial_value

    return func

In [11]:
ilr = 3.5e-4
# PPO parameters
model = PPO("MlpPolicy",
        env,
        learning_rate=ilr,
        verbose=1,
        tensorboard_log=f'{OUTPUT_PATH}/logdir'
        )

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [ ]:
# Train the agent for n steps
model.learn(total_timesteps=int(total_timesteps), callback=[checkpoint_callback, eval_callback])

# Save the trained agentP
model.save(f'{OUTPUT_PATH}/models/ppo_RL_BASELINE_{str(os_id.value)}_{total_timesteps:.1E}_{env_id.value}_{datetime.now().strftime("%d-%m_%H-%M-%S")}.zip')

Logging to /home/utente1/pighetti/HighwayDRL/training_output/logdir/PPO_8


/home/utente1/miniconda3/envs/highwayvenv/lib/python3.8/site-packages/stable_baselines3/common/evaluation.py:65: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Eval num_timesteps=1000, episode_reward=9.94 +/- 6.37
Episode length: 12.40 +/- 7.39
---------------------------------
| eval/              |          |
|    mean_ep_length  | 12.4     |
|    mean_reward     | 9.94     |
| time/              |          |
|    total_timesteps | 1000     |
---------------------------------
New best mean reward!
Eval num_timesteps=2000, episode_reward=5.08 +/- 2.51
Episode length: 6.80 +/- 2.71
---------------------------------
| eval/              |          |
|    mean_ep_length  | 6.8      |
|    mean_reward     | 5.08     |
| time/              |          |
|    total_timesteps | 2000     |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 27.3     |
|    ep_rew_mean     | 19.6     |
| time/              |          |
|    fps             | 4        |
|    iterations      | 1        |
|    time_elapsed    | 433      |
|    total_timesteps | 2048     |
---------------------------

### Continued learning

In [5]:
env_cl_id = 'dm-env-v0'
env_cl = gym.make(env_cl_id)

In [6]:
custom_objects = { 'learning_rate': 3.5e-4 }

model_cl = PPO.load(f"{OUTPUT_PATH}models/ppo_RL_sparseONLY_nolmap_[-1_1]WINDOWS_3.0E+04_multipleO-dm-env-v0_14-08_01-18-19", env=env_cl, custom_objects=custom_objects, tensorboard_log=f"{OUTPUT_PATH}logdir")

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [7]:
total_timesteps = 6e4

In [8]:
class TensorboardCallback(BaseCallback):
    def __init__(self, verbose=1):
        super(TensorboardCallback, self).__init__(verbose)
        self.coll_rew = 0
        self.rml_rew = 0
        self.high_speed_rew = 0
        self.current_steps = 0

    def _on_step(self) -> bool:        
        self.rml_rew += self.training_env.get_attr('rml_reward')[0]
        self.high_speed_rew += self.training_env.get_attr('high_speed_reward')[0]
            
        if self.locals["dones"][0]:
            self.coll_rew += -self.training_env.get_attr('collision_reward')[0]
            self.logger.record("eval/coll_rew", self.coll_rew)
            self.logger.record("episode/RML_rew", self.rml_rew / self.current_steps)
            self.logger.record("episode/high_speed_rew", self.high_speed_rew / self.current_steps)
            self.rml_rew = self.high_speed_rew = self.current_steps = 0
        else:
            self.current_steps = self.training_env.get_attr('steps')[0]
        return True

# Save a checkpoint every 1000 steps
checkpoint_callback = CheckpointCallback(save_freq=5000, save_path=f'{OUTPUT_PATH}checkpoints',
                                         name_prefix='dm_rl_callback')

eval_callback = EvalCallback(env_cl, best_model_save_path=f'{OUTPUT_PATH}/eval_logs',
                             log_path=f'{OUTPUT_PATH}/eval_logs', eval_freq=1000,
                             deterministic=True, render=False)

In [9]:
new_timesteps = 6e4

model_cl.learn(total_timesteps=int(new_timesteps), callback=[checkpoint_callback, eval_callback, TensorboardCallback()])

model_cl.save(f'{OUTPUT_PATH}models/ppo_RL_sparseONLY_nolmap_[-1_1]_CL_{str(os_id.value)}_{(total_timesteps + new_timesteps):.1E}_{env_cl_id}_{datetime.now().strftime("%d-%m_%H-%M-%S")}.zip')

Logging to C:/Users/pigo/Desktop/HighwayDRL/training_output/logdir\PPO_20


c:\Users\pigo\miniconda3\envs\highway\lib\site-packages\stable_baselines3\common\evaluation.py:65: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Eval num_timesteps=1000, episode_reward=-0.41 +/- 0.71
Episode length: 68.40 +/- 27.58
---------------------------------
| episode/           |          |
|    RML_rew         | 0.0134   |
|    high_speed_rew  | 0.164    |
| eval/              |          |
|    coll_rew        | 0        |
|    mean_ep_length  | 68.4     |
|    mean_reward     | -0.406   |
| time/              |          |
|    total_timesteps | 1000     |
---------------------------------
New best mean reward!
Eval num_timesteps=2000, episode_reward=-0.57 +/- 0.27
Episode length: 74.60 +/- 27.64
---------------------------------
| episode/           |          |
|    RML_rew         | 0.0714   |
|    high_speed_rew  | 0.32     |
| eval/              |          |
|    coll_rew        | 1        |
|    mean_ep_length  | 74.6     |
|    mean_reward     | -0.57    |
| time/              |          |
|    total_timesteps | 2000     |
---------------------------------
---------------------------------
| rollout/           |

KeyboardInterrupt: 

In [ ]:
model_cl.save(f'{OUTPUT_PATH}models/ppo_RL_sparseONLY_nolmap_[-1_1]_CL_{str(os_id.value)}_{(total_timesteps + new_timesteps):.1E}_{env_cl_id}_{datetime.now().strftime("%d-%m_%H-%M-%S")}.zip')